In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification

In [18]:
X, y = make_classification(n_samples=75,n_features=2,n_redundant=0,n_informative=2,n_clusters_per_class=1,random_state=42)

cgpa = 4 + (X[:,0]-X[:,0].min())/(X[:,0].max()-X[:,0].min())*6
profile_score = 40 + (X[:,1]-X[:,1].min())/(X[:,1].max()-X[:,1].min())*60

df = pd.DataFrame({"cgpa":cgpa,"profile_score":profile_score,"placed":y})

df["cgpa"] = (df["cgpa"]-df["cgpa"].mean())/df["cgpa"].std()
df["profile_score"] = (df["profile_score"]-df["profile_score"].mean())/df["profile_score"].std()

In [19]:
def initialize_parameters(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [20]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [21]:
def linear_forward(A_prev,W,b):

    Z = np.dot(W.T,A_prev)+b
    A = sigmoid(Z)

    return A

In [22]:
def L_layer_forward(X,parameters):

    A = X
    L = len(parameters)//2

    for l in range(1,L+1):

        A_prev = A

        W = parameters["W"+str(l)]
        b = parameters["b"+str(l)]

        A = linear_forward(A_prev,W,b)

    return A,A_prev

In [23]:
def update_parameters(parameters,y,y_hat,A1,X):

    W2_old = parameters["W2"].copy()

    dz2 = y_hat-y

    dw2 = dz2*A1
    db2 = dz2

    dA1 = W2_old*dz2
    dz1 = dA1*A1*(1-A1)

    dw1_col0 = dz1[0][0]*X
    dw1_col1 = dz1[1][0]*X

    lr = 0.001

    parameters["W2"][0][0] -= lr*dw2[0][0]
    parameters["W2"][1][0] -= lr*dw2[1][0]
    parameters["b2"][0][0] -= lr*db2

    parameters["W1"][0][0] -= lr*dw1_col0[0][0]
    parameters["W1"][1][0] -= lr*dw1_col0[1][0]
    parameters["b1"][0][0] -= lr*dz1[0][0]

    parameters["W1"][0][1] -= lr*dw1_col1[0][0]
    parameters["W1"][1][1] -= lr*dw1_col1[1][0]
    parameters["b1"][1][0] -= lr*dz1[1][0]

In [24]:
X_sample = df[["cgpa","profile_score"]].values[0].reshape(2,1).astype(np.float64)
y_sample = df["placed"].values[0]

In [25]:
params = initialize_parameters([2,2,1])

W1_before = params["W1"].copy()
b1_before = params["b1"].copy()
W2_before = params["W2"].copy()
b2_before = params["b2"].copy()

In [26]:
y_hat,A1 = L_layer_forward(X_sample,params)
y_hat_val = y_hat[0][0]

In [27]:
update_parameters(params,y_sample,y_hat_val,A1,X_sample)

lr = 0.001

your_dW1 = (params["W1"]-W1_before)/lr
your_db1 = (params["b1"]-b1_before)/lr
your_dW2 = (params["W2"]-W2_before)/lr
your_db2 = (params["b2"]-b2_before)/lr

In [28]:
W1_tf = tf.Variable(W1_before,dtype=tf.float64)
b1_tf = tf.Variable(b1_before,dtype=tf.float64)

W2_tf = tf.Variable(W2_before,dtype=tf.float64)
b2_tf = tf.Variable(b2_before,dtype=tf.float64)

In [29]:
x_tf = tf.constant(X_sample,dtype=tf.float64)
y_tf = tf.constant([[y_sample]],dtype=tf.float64)

In [30]:
with tf.GradientTape() as tape:

    Z1 = tf.matmul(tf.transpose(W1_tf),x_tf) + b1_tf
    A1 = tf.math.sigmoid(Z1)

    Z2 = tf.matmul(tf.transpose(W2_tf),A1) + b2_tf
    y_pred = tf.math.sigmoid(Z2)

    loss = tf.keras.losses.binary_crossentropy(
        y_tf,
        y_pred,
        from_logits=False
    )

In [31]:
grads = tape.gradient(loss,[W1_tf,b1_tf,W2_tf,b2_tf])

tf_dW1,tf_db1,tf_dW2,tf_db2 = [g.numpy() for g in grads]

In [33]:
print("W1 diff :",np.max(np.abs(your_dW1-(-tf_dW1))))
print("b1 diff :",np.max(np.abs(your_db1-(-tf_db1))))
print("W2 diff :",np.max(np.abs(your_dW2-(-tf_dW2))))
print("b2 diff :",np.max(np.abs(your_db2-(-tf_db2))))

W1 diff : 5.636116573448646e-15
b1 diff : 0.0
W2 diff : 3.2751579226442118e-15
b2 diff : 0.0
